# voicebox — Colab runner

Thin runner. All real code lives in the git repo; this notebook just rents a GPU.

**Steps:** mount Drive → clone repo → install deps → run scripts.

Set the GitLab repo URL in the second cell before first run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Persistent project root on Drive — survives session disconnects.
import os
PROJECT_ROOT = '/content/drive/MyDrive/voicebox'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
!pwd

In [ ]:
# Clone (first run) or pull (subsequent runs).
REPO_URL = 'git@gitlab.com:YOUR_USER/voicebox.git'  # <-- set this
REPO_DIR = f'{PROJECT_ROOT}/repo'

import os
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

os.chdir(REPO_DIR)
!pwd && git log -1 --oneline

In [ ]:
# Install package in editable mode. Colab Pro already has torch + CUDA.
!pip install -q -e .

In [ ]:
# Sanity check: GPU visible, voicebox stack instantiates, one forward pass.
!python scripts/check_env.py

In [ ]:
# Phase 1: extract concept vectors from the frozen teacher.
# Persist outputs to Drive so they survive a runtime disconnect.
VECTORS_OUT = f'{PROJECT_ROOT}/data/vectors/qwen25_7b.pt'
PROMPTS_IN  = f'{PROJECT_ROOT}/data/raw/synthetic.jsonl'

!mkdir -p $(dirname $VECTORS_OUT) $(dirname $PROMPTS_IN)

!python scripts/extract_vectors.py \
    --prompts $PROMPTS_IN \
    --out $VECTORS_OUT \
    --model Qwen/Qwen2.5-7B-Instruct \
    --dtype bfloat16 \
    --batch-size 8